In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)

        self.fc2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)

        self.dropout = nn.Dropout(dropout)

        # Projection if dimensions differ
        self.shortcut = nn.Sequential()
        if in_features != out_features:
            self.shortcut = nn.Sequential(
                nn.Linear(in_features, out_features),
                nn.BatchNorm1d(out_features)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.fc1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)
        out = self.bn2(out)

        out += identity
        out = F.relu(out)

        return out

In [ ]:
class WideResNetTabular(nn.Module):
    def __init__(self, input_dim, num_classes, width=256, depth=4, dropout=0.3):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.ReLU()
        )

        # Stack residual blocks
        self.blocks = nn.Sequential(
            *[ResidualBlock(width, width, dropout) for _ in range(depth)]
        )

        self.head = nn.Sequential(
            nn.Linear(width, width // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width // 2, num_classes)
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [ ]:
dft = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_t.csv')
dfn = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_n.csv')
dfm = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_m.csv')
dfv = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/validation_enc.csv")

In [ ]:
dfn.head(3)

,patient_filename,text,text_embedding_clinicalbert,text_embedding_biobert,text_embedding_pubmedbert,n
0,TCGA-05-4244.3a844132-f813-4d8e-8f7d-dbae0b23d7fd,Diagnosis: 1. Resection material from the righ...,"[0.1355716437101364, -0.12100697308778763, -0....","[0.04545216262340546, -0.03954649344086647, -0...","[-0.06522239744663239, 0.07896429300308228, -0...",2.0
1,TCGA-05-4245.902fe548-5b93-49c9-81db-2af4a4a88f3c,Diagnosis: 1. Atypical resection material from...,"[0.08271332830190659, -0.22827120125293732, -0...","[0.21559493243694305, -0.14149688184261322, -0...","[-0.05436353012919426, 0.07354100793600082, -0...",2.0
2,TCGA-05-4249.7e920317-d5c2-4160-9e2b-ef0101eb5a23,Diagnosis: 2. and 3. Two-part resection materi...,"[0.1757894605398178, -0.17660431563854218, -0....","[0.1323431432247162, -0.16728118062019348, -0....","[-0.11425544321537018, 0.08171500265598297, -0...",0.0


In [ ]:
yn = dfn['n']
Xn = dfn[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]
X_train, X_test, y_train, y_test = train_test_split(Xn, yn, test_size=0.25, stratify=yn)

In [ ]:
import ast

def parse_list_string(list_string):
    return ast.literal_eval(list_string)

X_train_list = []

for index, row in X_train.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_train_list.append(combined_vector)

X_train_n = np.array(X_train_list)
y_train_n = y_train.values

print(f"Shape of X_train: {X_train_n.shape}")
print(f"Shape of y_train: {y_train_n.shape}")

Shape of X_train: (3619, 2304)
Shape of y_train: (3619,)


In [ ]:
X_test_list_n = []
# text_embedding_clinicalbert, text_embedding_biobert, text_embedding_pubmedbert
for index, row in X_test.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_test_list_n.append(combined_vector)

X_test_n = np.array(X_test_list_n)
y_test_n = y_test.values

print(f"Shape of X_train: {X_test_n.shape}")
print(f"Shape of y_train: {y_test_n.shape}")

Shape of X_train: (1207, 2304)
Shape of y_train: (1207,)


In [ ]:
#y_train_t = y_train_t - y_train_t.min()
#y_test_t = y_test_t - y_test_t.min()
print(type(y_train_n[0]), y_train_n[:10])

<class 'numpy.float64'> [0. 1. 2. 0. 0. 2. 0. 3. 0. 0.]


In [ ]:
train_dataset = TabularDataset(X_train_n, y_train_n)
val_dataset = TabularDataset(X_test_n, y_test_n)

In [ ]:
num_classes = len(np.unique(y_train))
print("Unique classes:", np.unique(y_train))
print("Num classes:", num_classes)

Unique classes: [0. 1. 2. 3.]
Num classes: 4


In [ ]:
print("Unique labels:", set(y_train_n))
print("Num classes:", num_classes)
print("Max label:", max(y_train_n))

Unique labels: {np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)}
Num classes: 4
Max label: 3.0


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [ ]:
# width=512, 768
model = WideResNetTabular(
    input_dim=X_train_n.shape[1],
    num_classes=num_classes,
    width=768,
    depth=4,
    dropout=0.3
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

WideResNetTabular(
  (input_layer): Sequential(
    (0): Linear(in_features=2304, out_features=768, bias=True)
    (1): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (fc1): Linear(in_features=768, out_features=768, bias=True)
      (bn1): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=768, out_features=768, bias=True)
      (bn2): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (shortcut): Sequential()
    )
    (1): ResidualBlock(
      (fc1): Linear(in_features=768, out_features=768, bias=True)
      (bn1): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=768, out_features=768, bias=True)
      (bn2): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_run

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode  # "max" for metrics like F1/AUC
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improvement = (score - self.best_score) if self.mode == "max" else (self.best_score - score)

        if improvement > self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.early_stop = True

        return self.early_stop

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_classes, epochs=50):
    early_stopper = EarlyStopping(patience=7, mode="max")

    best_f1 = 0
    best_metrics = None
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        metrics = evaluate_metrics(model, val_loader, num_classes)

        print(f"Epoch {epoch+1}")
        print(f"Loss: {avg_loss:.4f}")
        print(f"F1 Macro: {metrics['f1_macro']:.4f}")
        print(f"F1 Weighted: {metrics['f1_weighted']:.4f}")
        print(f"AUC: {metrics['auc']:.4f}")
        print("-" * 40)

        # Track best model (based on F1 macro)
        if metrics["f1_macro"] > best_f1:
            best_f1 = metrics["f1_macro"]
            best_metrics = metrics
            best_epoch = epoch + 1

            # Optional: save best model
            torch.save(model.state_dict(), "best_model.pt")

        # Early stopping
        if early_stopper.step(metrics["f1_macro"]):
            print("Early stopping triggered.")
            break

    # Print best results at the end
    print("\n===== BEST MODEL =====")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best F1 Macro: {best_metrics['f1_macro']:.4f}")
    print(f"Best F1 Micro: {best_metrics['f1_micro']:.4f}")
    print(f"Best F1 Weighted: {best_metrics['f1_weighted']:.4f}")
    print(f"Best AUC: {best_metrics['auc']:.4f}")

In [ ]:
def evaluate_metrics(model, loader, num_classes):
    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            # Move input tensors to the same device as the model
            X_batch = X_batch.to(device)

            logits = model(X_batch)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # F1 scores
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    # AUC (multiclass)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except ValueError:
        auc = float("nan")  # happens if a class is missing in batch

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "auc": auc
    }

train_model(
            model, train_loader, val_loader, optimizer,\
            criterion, num_classes=num_classes, epochs=70
            )

Epoch 1
Loss: 1.0261
F1 Macro: 0.3083
F1 Weighted: 0.5703
AUC: 0.7745
----------------------------------------
Epoch 2
Loss: 0.8187
F1 Macro: 0.4075
F1 Weighted: 0.6383
AUC: 0.8097
----------------------------------------
Epoch 3
Loss: 0.7426
F1 Macro: 0.3806
F1 Weighted: 0.5551
AUC: 0.7961
----------------------------------------
Epoch 4
Loss: 0.6748
F1 Macro: 0.4175
F1 Weighted: 0.6517
AUC: 0.8507
----------------------------------------
Epoch 5
Loss: 0.6148
F1 Macro: 0.4345
F1 Weighted: 0.6350
AUC: 0.8382
----------------------------------------
Epoch 6
Loss: 0.5753
F1 Macro: 0.4153
F1 Weighted: 0.6214
AUC: 0.8167
----------------------------------------
Epoch 7
Loss: 0.5315
F1 Macro: 0.5258
F1 Weighted: 0.6686
AUC: 0.8308
----------------------------------------
Epoch 8
Loss: 0.4925
F1 Macro: 0.4808
F1 Weighted: 0.6557
AUC: 0.8477
----------------------------------------
Epoch 9
Loss: 0.4483
F1 Macro: 0.5108
F1 Weighted: 0.6482
AUC: 0.8414
----------------------------------------
E